# BrailleVision — YOLOv8 Braille Dot Detector Training

**Run these cells top-to-bottom to train a full Braille dot detection model on a free T4 GPU.**

Steps:
1. Install dependencies
2. Clone repository (or mount Drive with pre-uploaded files)
3. Download real Braille datasets + generate synthetic images
4. Merge & split into train/val/test
5. Train YOLOv8n
6. Evaluate & download weights

In [ ]:
# ── Cell 1: Check GPU ──────────────────────────────────────────
!nvidia-smi
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────
!pip install -q ultralytics opencv-python-headless numpy pyyaml

In [ ]:
# ── Cell 3: Get the BrailleVision project ─────────────────────
import os

# OPTION A: Upload the project as a zip from your PC
# from google.colab import files
# uploaded = files.upload()  # select BrailleVision.zip
# !unzip -q BrailleVision.zip

# OPTION B: Clone from GitHub (replace with your actual repo URL)
REPO_URL = 'https://github.com/YOUR_USERNAME/BrailleVision.git'  # <-- UPDATE THIS
if not os.path.exists('BrailleVision'):
    !git clone {REPO_URL}

%cd BrailleVision
!ls

In [ ]:
# ── Cell 4: Download real datasets ────────────────────────────
# Downloads Angelina Braille Reader dataset + DSBI dataset from GitHub
!python training/scripts/download_datasets.py --skip-synthetic
print('\nReal datasets downloaded.')

In [ ]:
# ── Cell 5: Generate synthetic training images ─────────────────
# Generates 800 realistic synthetic Braille images to supplement real data
!python training/scripts/generate_synthetic.py --count 800
print('\nSynthetic generation done.')

In [ ]:
# ── Cell 6: Merge all sources + create 70/20/10 split ─────────
!python training/scripts/merge_and_split.py

# Verify
!python training/scripts/verify_dataset.py

# Show data.yaml
!cat dataset/data.yaml

In [ ]:
# ── Cell 7: Train YOLOv8n ─────────────────────────────────────
# Uses GPU (device=0), 100 epochs, batch 16
# Expected time: ~45-90 mins on T4 GPU
!python training/scripts/train.py \
    --epochs 100 \
    --batch 16 \
    --device 0 \
    --name braille_dot_v1

In [ ]:
# ── Cell 8: Evaluate trained model ────────────────────────────
!python training/scripts/evaluate.py --skip-e2e
!cat training/runs/eval_report.md

In [ ]:
# ── Cell 9: Download trained weights ──────────────────────────
from google.colab import files
import os

# Download best.pt
if os.path.exists('model/best.pt'):
    files.download('model/best.pt')
    print('Downloaded best.pt')
else:
    print('best.pt not found - check training logs above')

# Download ONNX (for CPU inference)
if os.path.exists('model/model.onnx'):
    files.download('model/model.onnx')
    print('Downloaded model.onnx')

print('\nPlace these files in the model/ folder of your local BrailleVision project.')